In [1]:
%env PYTORCH_ENABLE_MPS_FALLBACK=1
%cd ..

env: PYTORCH_ENABLE_MPS_FALLBACK=1
/Users/n-zagainov/kolobok


In [2]:
import json
from pathlib import Path

import numpy as np
import cv2

from matplotlib import pyplot as plt, colormaps as cm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.io import read_image
from torchvision import transforms
from torchvision.transforms import functional as VF, InterpolationMode

from tqdm import tqdm

from transformers import SegformerForSemanticSegmentation, SegformerConfig

images_dir = Path("data/dataset_crop")
labels_path = Path("data/result.json")

In [3]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [4]:
with open(labels_path, "r") as f:
    labels = json.load(f)

for i in range(len(labels["images"])):
    name = labels["images"][i]["file_name"]
    name = images_dir / name.split("__")[-1]
    labels["images"][i]["file_name"] = name

In [5]:
class SegmentationDataset(Dataset):
    def __init__(
        self,
        labels: dict,
        prune_empty: bool = True,
        image_transform: nn.Module = None,
        mask_transform: nn.Module = None,
    ):
        self.labels = labels
        self.image_transform = image_transform
        self.mask_transform = mask_transform

        for img in labels["images"]:
            assert Path(img["file_name"]).exists()

        images = [None] * len(self.labels["images"])
        annotations = [[] for _ in range(len(self.labels["images"]))]

        for image in labels["images"]:
            images[image["id"]] = image["file_name"]

        for annotation in labels["annotations"]:
            annotations[annotation["image_id"]].append(
                {
                    "poly": np.array(
                        annotation["segmentation"], dtype=np.int32
                    ).reshape(-1, 2),
                    "category": annotation["category_id"] + 1,
                }
            )

        self.images = images
        self.annotations = annotations

        if prune_empty:
            ids = [i for i in range(len(self.labels["images"])) if annotations[i]]
            self.images = [self.images[i] for i in ids]
            self.annotations = [self.annotations[i] for i in ids]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx: int):
        image = read_image(self.images[idx]) / 255.0
        mask = np.zeros(list(image.shape[1:]), dtype=np.uint8)

        for annotation in self.annotations[idx]:
            mask = cv2.fillPoly(
                mask, [annotation["poly"]], color=annotation["category"]
            )
        mask = torch.tensor(mask, dtype=torch.uint8)[None]

        if self.image_transform:
            image = self.image_transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return image.contiguous(), mask.squeeze(0).contiguous()


transform_image = transforms.Compose(
    [
        transforms.Resize((512, 512), interpolation=InterpolationMode.BICUBIC),
    ]
)

transform_mask = transforms.Compose(
    [
        transforms.Resize((512, 512), interpolation=InterpolationMode.NEAREST),
        transforms.Lambda(lambda x: x.long()),
    ]
)


In [6]:
dataset = SegmentationDataset(
    labels, image_transform=transform_image, mask_transform=transform_mask
)
print(f"total number of images: {len(dataset)}")

train_dataset, val_dataset = random_split(dataset, [0.75, 0.25])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

total number of images: 160


In [7]:
class SegformerWrapper(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def _upscale_logits(self, logits: torch.Tensor, target_shape: tuple = (512, 512)):
        logits = VF.resize(
            logits, 
            size=target_shape, 
            interpolation=InterpolationMode.BILINEAR,
        )
        return logits

    def forward(self, images: torch.Tensor):
        logits = self.model(images).logits
        logits = self._upscale_logits(logits, target_shape=images.shape[2:])
        return logits

config = SegformerConfig.from_pretrained("nvidia/segformer-b2-finetuned-ade-512-512")
config.num_labels = 6
segformer = SegformerForSemanticSegmentation(config)

model = SegformerWrapper(segformer)



In [8]:
def dice_loss(logits: torch.Tensor, labels: torch.Tensor, eps=1e-6):
    """
    logits: [B, C, H, W] raw scores
    labels: [B, H, W] ints in [0..C-1]
    """
    # convert to one-hot: [B, C, H, W]
    num_classes = logits.shape[1]
    labels_onehot = F.one_hot(labels, num_classes).permute(0, 3, 1, 2).float()
    probs = F.softmax(logits, dim=1)
    # compute per-class dice
    intersection = torch.sum(probs * labels_onehot, dim=(2,3))
    union = torch.sum(probs + labels_onehot, dim=(2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()

def ce_loss(logits: torch.Tensor, labels: torch.Tensor):
    """
    logits: [B, C, H, W] raw scores
    labels: [B, H, W] ints in [0..C-1]
    """
    return F.cross_entropy(logits, labels)

class SegformerLoss(nn.Module):
    def __init__(self, ce_weight: float = 1, dice_weight: float = 1):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight  

    def forward(self, logits: torch.Tensor, labels: torch.Tensor):
        ce = ce_loss(logits, labels)
        dice = dice_loss(logits, labels)
        return self.ce_weight * ce + self.dice_weight * dice

In [9]:
def train_fn(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    n_epochs: int = 5,
):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.1,
        patience=3,
    )
    loss_fn = SegformerLoss(ce_weight=1, dice_weight=1)

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0
        train_loop = tqdm(train_loader, desc=f"[{epoch}/{n_epochs}] Train Epoch")

        for i, (x, y) in enumerate(train_loop):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            preds_raw = model(x)
            loss = loss_fn(preds_raw, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            train_loop.set_postfix(loss=running_loss / (i + 1))

        model.eval()
        running_loss = 0
        eval_loop = tqdm(val_loader, desc=f"[{epoch}/{n_epochs}] Evaluation Epoch")

        with torch.no_grad():
            for i, (x, y) in enumerate(eval_loop):
                x = x.to(device)
                y = y.to(device)

                preds_raw = model(x)
                loss = loss_fn(preds_raw, y)
                running_loss += loss.item()

                eval_loop.set_postfix(loss=running_loss / (i + 1))

        scheduler.step(metrics=running_loss / (i + 1))


In [10]:
config = SegformerConfig.from_pretrained("nvidia/segformer-b2-finetuned-ade-512-512")
config.num_labels = 6
segformer = SegformerForSemanticSegmentation(config)

model = SegformerWrapper(segformer)



In [11]:
def get_num_params(model: nn.Module):
    return sum(param.numel() for param in model.parameters())

print(f"number of model parameters: {get_num_params(model)}")

number of model parameters: 27351238


In [12]:
train_fn(
    model,
    train_loader,
    val_loader,
    get_device(),
    n_epochs=50,
)

[1/50] Train Epoch:   0%|          | 0/30 [00:00<?, ?it/s]/Users/n-zagainov/.local/share/mamba/envs/klbk/lib/python3.11/site-packages/torch/autograd/graph.py:824: UserWarning: The operator 'aten::_upsample_bilinear2d_aa_backward.grad_input' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:14.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[1/50] Train Epoch:   0%|          | 0/30 [00:00<?, ?it/s]


RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [13]:
model = model.to(get_device())
# imgs, masks = next(iter(val_loader))
img, mask = dataset[0]
imgs = img[None]  # add batch dimension
masks = mask[None]  # add batch dimension

imgs = imgs.to(get_device())
masks = masks.to(get_device())

loss_fn = SegformerLoss(ce_weight=1, dice_weight=1)


In [16]:
masks.shape

torch.Size([1, 1143, 1103])

In [14]:
logits = model(imgs)



In [17]:
loss = F.cross_entropy(logits, masks)

In [18]:
loss.backward()

In [ ]:
# model.eval()
# img, gt_mask = val_dataset[5]
# pred = model(img[None].to(get_device())).logits
# # pred[:, 1:] *= 5
# pred_mask = torch.argmax(pred.cpu().squeeze(), dim=0).numpy()

# fig, axes = plt.subplots(ncols=2, figsize=(10, 10))

# axes[0].imshow(img.permute(1, 2, 0))
# cmap = cm.get_cmap("hsv")
# pred_mask = np.where(pred_mask[..., None] != 0, cmap(pred_mask / 6)[..., 1:], 0)
# axes[1].imshow(pred_mask)


# for ax in axes:
#     ax.axis("off")

# plt.show()

AttributeError: 'Tensor' object has no attribute 'logits'

In [13]:
pred.mean(dim=(0, 2, 3))

tensor([-0.4997, -1.4175, -4.9606, -0.8412,  3.3687, -4.9106], device='cuda:0',
       grad_fn=<MeanBackward1>)